In [ ]:
import pandas as pd
import numpy as np

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

def extract_fasta_headers(fasta_file):
    headers = []
    with open(fasta_file, "r") as file:
        for line in file:
            if line.startswith(">"):
                headers.append(line[1:].split()[0])  # Store the header without newline
    return headers

In [ ]:
fasta_file = "minnow_index_mm10_TE/ref_k101_fixed.fa"  

headers = extract_fasta_headers(fasta_file)
headers[1:10]
print("Number of loci:")
len(headers)
loci=pd.Series(headers)

In [ ]:
df_withAge = pd.read_csv("/mnt/TEdata/snakemake_TE_full/computeAge_mm10/output.txt", sep='\t', header=None)
df_withAge.columns = ["seqname", "start", "end", "subfamily", "family", "class", "strand", "propSub", "JCdist"]
df_withAge["mya"] = (df_withAge.JCdist * 100) / (4.5 * 2 * 100) * 1000
df_withAge["ageClass"] = np.where(df_withAge['mya'] < 2, 'young', 'old')

# remove all "?"
df_withAge = df_withAge.replace("\\?", "", regex=True)
# remove unwanted classes
df_withAge = df_withAge.loc[~df_withAge['class'].isin(["Simple_repeat" "Low_complexity", "Other", "Satellite", "Unknown", "RNA"])]

df_withAge.mya.describe()

df_withAge.ageClass.value_counts()

df_withAge['class'].value_counts()

df_withAge.family.value_counts()

df_withAge.subfamily.value_counts()

In [ ]:
# read gtf file
gtf_columns = ["seqname", "source", "feature", "start", "end", 
            "score", "strand", "frame", "attribute"]

gtf = pd.read_csv("data/TEannotation/mm10_GENCODE_rmsk_TE.gtf.gz", 
    header=None, compression="gzip", sep="\t", names=gtf_columns)

# Function to parse the 'attribute' column
def parse_attributes(attribute_string):
    """Parses the attribute column of a GTF file into a dictionary."""
    attributes = {}
    for attr in attribute_string.strip().split(";"):
        if attr.strip():
            key, value = attr.strip().split(" ", 1)
            attributes[key] = value.strip('"')  # Remove surrounding quotes
    return attributes

# Apply the parsing function and expand into separate columns
attribute_df = gtf['attribute'].apply(parse_attributes).apply(pd.Series)

# Merge with the main DataFrame
gtf = gtf.drop(columns=['attribute']).join(attribute_df)

# subtract 1 from start column
gtf.start = gtf.start - 1

gtf.to_csv("gtf.csv")

In [ ]:
gtf = pd.read_csv("gtf.csv", index_col=0)

headers = pd.Series(headers)

(~ headers.isin(gtf.transcript_id)).sum()  # 0 if they are all present

gtf_present = gtf.loc[gtf.transcript_id.isin(headers),:]
gtf_present
gtf_present = gtf_present.merge(df_withAge, on=['seqname', 'start', 'end', 'strand'])

selected_cols = ['seqname', 'start', 'end', 'score', 'strand','transcript_id','subfamily','family','class','propSub', 'JCdist', 'mya', 'ageClass']

gtf_present = gtf_present.loc[:, gtf_present.columns.intersection(selected_cols)]

gtf_present.mya.describe()

gtf_present.ageClass.value_counts()

gtf_present['class'].value_counts()

gtf_present.family.value_counts()

len(gtf_present.family.unique())

gtf_present.subfamily.value_counts()

In [ ]:
rmsk = pd.read_csv("/mnt/TEdata/TEbenchmarking/data/TEannotation/mm10_rmsk.bed", sep = "\t", header=None)
rmsk.columns=["seqname","start","end","name","score","strand"]
rmsk.start=rmsk.start-1

In [ ]:

gtf_present_rmsk = gtf_present.merge(rmsk, on=['seqname', 'start', 'end', 'strand'])
gtf_present_rmsk # same dimensions, all loci are present in the rmsk annotation
gtf_old = gtf_present.loc[gtf_present.ageClass == "old", :]
gtf_old
families_old = gtf_old.family.value_counts()[gtf_old.family.value_counts()>50].index

In [ ]:

selected_loci_old_list = []
for family in families_old:
    loci = gtf_old.loc[gtf_old.family == family, "transcript_id"]
    loci_sub = loci.sample(n=min(100, len(loci)), random_state = 213, replace=False)
    selected_loci_old_list.append(loci_sub)
# Concatenate all samples at the end
selected_loci_old = pd.concat(selected_loci_old_list, ignore_index=True)

gtf_young = gtf_present.loc[gtf_present.ageClass == "young", :]
families_young = gtf_young.family.value_counts()[gtf_young.family.value_counts()>50].index
selected_loci_young_list = []
for family in families_young:
    loci = gtf_young.loc[gtf_young.family == family, "transcript_id"]
    loci_sub = loci.sample(n=min(300, len(loci)), random_state = 213, replace=False)
    selected_loci_young_list.append(loci_sub)
# Concatenate all samples at the end
selected_loci_young = pd.concat(selected_loci_young_list, ignore_index=True)
selected_loci_young
selected_loci_old.isin(headers).value_counts()
selected_loci_young.isin(headers).value_counts()
selected_loci_old.isin(gtf_present_rmsk.transcript_id).value_counts()

selected_loci_young.isin(gtf_present_rmsk.transcript_id).value_counts()
gtf_present_rmsk.loc[gtf_present_rmsk.transcript_id.isin(selected_loci_young),:]
selected_loci_old.to_csv("selected_loci_old.csv", header=False, index=False)
selected_loci_young.to_csv("selected_loci_young.csv", header=False, index=False)
